# Aula 18 — Grafos de Logística de Insumos e Produtos Acabados

**UNIFEI • ECAA08 Automática • Grupo 2**
**Projeto:** SCADA da Máquina Pneumática de Envasamento de Água em Copos.

## 1. Situação-problema e adaptação do roteiro

A máquina recebe copos, água própria para consumo, tampas e ar comprimido. O copo passa por dispensa, envase de 150 ml, posicionamento de tampa, termosselagem e ejeção para a esteira. A Aula 18 amplia a visão para recebimento, abastecimento, estoque e expedição, separando **fluxo de materiais** de **circulação de veículos internos**.

O roteiro do professor usa uma fábrica de fertilizantes. Aqui o mesmo princípio matemático é aplicado à envasadora. Embalagem secundária, estoques externos, portaria, doca e corredores são **extensões didáticas propostas**, pois o projeto-base não fornece um layout fabril medido. Não são instalações concluídas nem novas tags físicas. Os comprimentos são hipóteses explícitas para comparar rotas.

| Elemento do projeto | Representação nesta entrega |
|---|---|
| Copos plásticos | Recebimento → estoque → MAG-100 → E1 |
| Água | Recebimento → TK-101 → E2 |
| Tampas | Recebimento → estoque → E3 |
| Ar comprimido | Utilidade necessária; não é ingrediente do produto nem via de veículo |
| Cinco estações | E1 dispensa → E2 envase → E3 tampa → E4 selagem → E5 ejeção |
| Esteira M-002 | EST-500_Saida → embalagem → estoque de produto → doca |

## 2. Dois modelos para perguntas diferentes

| Camada | Vértices/arestas | Peso | Pergunta |
|---|---|---|---|
| $G_M$: materiais | Recebimentos, estoques e etapas de transformação | Distância estimada em m | Por onde um insumo se incorpora ao processo e segue até expedição? |
| $G_V$: veículos | Áreas acessíveis e corredores autorizados | Distância em m; modelo separado para tempo em min | Por onde o veículo interno pode circular? |

Uma ligação entre E2 e E3 representa a progressão do copo na mesa, não uma rua. O veículo não atravessa dosador, prensa ou cabeçote. Identificadores de camadas diferentes não são intercambiáveis. Uma camada específica de alocação de estoque seria uma extensão futura; o entregável atual implementa os dois modelos centrais solicitados.

## 3. Grafo de materiais

```mermaid
flowchart LR
    RC[Recebimento copos] --> EC[Estoque copos] --> MG[MAG-100] --> E1[Dispensa]
    RA[Recebimento água] --> TK[TK-101 Água] --> E2[Envase 150 ml]
    E1 --> E2 --> E3[Tampa] --> E4[Selagem] --> E5[Ejeção]
    RT[Recebimento tampas] --> ET[Estoque tampas] --> E3
    E5 --> ES[EST-500 Saída] --> EM[Embalagem] --> EP[Estoque produto] --> DO[Doca expedição]
```

| Origem | Destino | Distância estimada (m) |
|---|---|---:|
| RECEB_COPOS | ESTOQUE_COPOS | 8 |
| ESTOQUE_COPOS | MAG-100_Copos | 6 |
| MAG-100_Copos | E1_DISPENSA | 1 |
| RECEB_AGUA | TK-101_Agua | 10 |
| TK-101_Agua | E2_ENVASE | 2,7 |
| RECEB_TAMPAS | ESTOQUE_TAMPAS | 7 |
| ESTOQUE_TAMPAS | E3_TAMPA | 5 |
| E1→E2; E2→E3; E3→E4; E4→E5 | Cada avanço lógico | 0,6 por trecho |
| E5_EJECAO | EST-500_Saida | 0,8 |
| EST-500_Saida | EMBALAGEM | 3 |
| EMBALAGEM | ESTOQUE_PRODUTO | 5 |
| ESTOQUE_PRODUTO | DOCA_EXPEDICAO | 8 |

O valor 2,7 m mantém a referência nominal de 270 cm até o dosador da Aula 14, agregada ao abastecimento de E2. Os demais valores são estimativas didáticas; em particular, 0,6 m não é uma medição do passo angular da mesa. O modelo logístico agrega a rede hidráulica, sem repetir sua redundância interna.

**Cuidado semântico:** um caminho de água começa em E2 e um de tampas em E3 porque esses insumos se incorporam nesses pontos. Isso não autoriza fabricar um copo pulando E1 ou E2. A rota do copo obrigatoriamente contém E1–E5 em ordem. Convergência de arestas é uma possibilidade de transferência; não representa sozinha uma operação AND, balanço de massa ou sincronismo. O notebook demonstra separadamente a disponibilidade conjunta de copos, água, tampas e ar. Essa verificação não substitui estoque quantitativo, rastreabilidade sanitária ou permissivos do controlador.

### 3.1. Propriedade DAG

O fluxo normal modelado não retorna a uma etapa anterior: é um grafo acíclico dirigido. A implementação de Kahn verifica essa propriedade em $O(|V|+|E|)$ e produz uma ordem topológica. Para toda aresta $(u,v)$, sua posição satisfaz $ord(u)<ord(v)$.

A mesa física gira ciclicamente, mas os vértices deste grafo representam o progresso de uma unidade de produto, não a posição periódica do mecanismo. Retrabalho, reciclagem ou devolução exigiriam modelagem adicional; um processo industrial não é necessariamente DAG em qualquer abstração.

## 4. Circulação interna de veículos

```mermaid
flowchart LR
    P[Portaria] --> R[Recebimento]
    R <-->|10 m| A[Corredor A]
    A <-->|8 m| AB[Abastecimento]
    A <-->|12 m| B[Corredor B]
    B <-->|8 m| EF[Estoque final]
    EF -->|10 m| D[Doca]
    R -->|30 m| X[Via externa] -->|35 m| D
    D --> S[Saída] --> P
```

O trecho Portaria→Recebimento tem 20 m; Doca→Saída, 15 m; Saída→Portaria, 10 m. Cada sentido de uma via de mão dupla é uma aresta explícita. Vias sem reverso cadastrado são de sentido único. O anel de circulação contém ciclos legítimos e não deve ser submetido à exigência de aciclicidade do fluxo normal de materiais.

O modelo descreve um veículo/carrinho de movimentação interna em uma área hipotética da instalação. Ele não supõe que um caminhão possa entrar na máquina. Compatibilidade de veículo, largura, capacidade e segregação de pessoas dependem de um layout real e não são validadas por estes pesos.

## 5. Menor caminho e exemplos resolvidos

A classe `GrafoLogistico` cadastra conexões dirigidas e ponderadas e usa Dijkstra para minimizar $\sum w(u,v)$, com pesos finitos não negativos. O notebook inclui toda a implementação, sem depender de outros arquivos. Bloqueios de nós/trechos, retorno `(inf, [])` e rejeição de entradas inválidas seguem a Aula 14. A complexidade usual é $O((|V|+|E|)\log|V|)$ por consulta.

**Copo até expedição:**

$$8+6+1+4(0{,}6)+0{,}8+3+5+8=34{,}2\text{ m}.$$

A sequência é RECEB_COPOS → ESTOQUE_COPOS → MAG-100 → E1 → E2 → E3 → E4 → E5 → ESTEIRA → EMBALAGEM → ESTOQUE_PRODUTO → DOCA. O validador verifica adjacência, soma dos pesos e presença das cinco operações. Há uma única sequência permitida de fabricação; Dijkstra não inventa alternativas tecnicamente proibidas.

**Veículo do recebimento até doca:** a rota pelos corredores custa $10+12+8+10=40$ m. A via externa custa $30+35=65$ m. Sem restrições, escolhemos 40 m. Bloqueado o corredor B, escolhemos 65 m. Bloqueados B e via externa, não existe rota.

## 6. Investigações resolvidas no notebook

1. **Rotas até estoques:** as consultas dos três recebimentos até seus estoques locais custam 8 m para copos, 10 m para água e 7 m para tampas. Também se apresentam as rotas completas até expedição: 34,2 m, 31,3 m e 30 m, respectivamente. As diferenças refletem o ponto de incorporação; não são comparação entre três receitas alternativas de produto.
2. **Armazenagem indisponível:** bloquear EMBALAGEM→ESTOQUE_PRODUTO torna a rota do copo até a doca inviável. O modelo não cria um atalho contornando armazenagem ou qualidade.
3. **Mão dupla para mão única:** remover CORREDOR_B→CORREDOR_A preserva A→B. O notebook valida que ainda existe o anel Portaria→Recebimento→A→B→Estoque→Doca→Saída→Portaria. Um caminho de um vértice até ele mesmo, de custo zero, não é usado como prova de existência de ciclo.
4. **Distância versus tempo:** em outro grafo, todos os custos são convertidos para minutos. Com velocidades hipotéticas de 2 m/min nos corredores e 10 m/min na via externa, a rota curta custa 20 min e a externa 6,5 min. O caminho mais curto em metros não é o mais rápido. As unidades nunca se somam na mesma consulta.
5. **Quarentena:** um submodelo RECEBIMENTO→QUARENTENA→ESTOQUE_LIBERADO começa com a saída bloqueada. Somente aprovação booleana explícita libera a transferência. Reprovação posterior volta a bloqueá-la. Isso representa a decisão de qualidade, não um resultado de Dijkstra.

## 7. Critérios de aceitação e execução

| Critério | Evidência executável |
|---|---|
| Dois grafos coerentes e separados | `criar_fluxo_materiais()` e `criar_circulacao_veiculos()` |
| Origem recebimento e destino expedição | Três consultas de insumo e validação completa da rota do copo |
| Respeito às operações obrigatórias | Asserts de ordem E1–E5 e bloqueio individual de cada estação |
| Pesos e distâncias consistentes | Soma independente das arestas pelo validador |
| DAG de materiais | Kahn e verificação de todas as arestas na ordem topológica |
| Sentidos e contingências | Remoção de reverso, rota alternativa, duplo bloqueio e restauração |
| Custo homogêneo e qualidade | Grafos separados m/min e liberação/rebloqueio da quarentena |

Abra o notebook de mesmo nome em Jupyter ou Colab, selecione Python 3.10 ou superior e execute todas as células em ordem. Somente biblioteca padrão; sem instalação adicional, imagens externas ou conexão de rede durante a execução. As saídas gravadas foram produzidas por execução local das células em ordem; reexecute no seu ambiente. Os testes usam `assert`, portanto não devem ser executados com `python -O`.

## 8. Limitações e conclusão

Este é um modelo computacional didático. Não demonstra implantação física, comunicação SCADA/Automation Studio, capacidade logística, layout medido, lotes reais ou controle industrial validado. Os valores de distância/velocidade devem ser substituídos por medições se houver uma aplicação real. Não foram alterados arquivos de outras aulas.

O entregável implementa os dois grafos, Dijkstra, a rota completa de recebimento à expedição, validação estrutural da sequência e as investigações adaptadas ao projeto. Separar as camadas permite discutir abastecimento e circulação sem confundir tubulação, trajetória do copo e caminho do veículo.

## 9. Referências e rastreabilidade

- [Roteiro da Aula 18 do professor](https://github.com/valerio-unifei/ECAA08-2026.2-Projeto/blob/c70e838aac076856727d0bb2151d20ff23d08600/etapa-02-grafos/18%20-%20Grafos%20de%20Logistica%20de%20Insumos%20e%20Produtos%20Acabados.md).
- [Descrição técnica da envasadora — Grupo 2](https://github.com/poggerssLL/Automatica---Grupo-2/blob/b25ada3f919adb3011a792fe62736cc941ac6c23/README.md).
- [Insumos do Grupo 2](https://github.com/poggerssLL/Automatica---Grupo-2/blob/b25ada3f919adb3011a792fe62736cc941ac6c23/etapa-01-logica/01%20-%20Insumos.md).


## Estrutura de grafo e Dijkstra

In [1]:
from math import inf, isfinite, isclose
from heapq import heappush, heappop
from numbers import Real


class GrafoDirigido:
    """Grafo simples dirigido: custos finitos não negativos e unidade única."""
    def __init__(self, vertices, unidade):
        vertices = tuple(vertices)
        if not vertices or any(not isinstance(v, str) or not v for v in vertices):
            raise ValueError("Informe vértices textuais não vazios")
        if len(set(vertices)) != len(vertices):
            raise ValueError("Vértices duplicados")
        if not isinstance(unidade, str) or not unidade:
            raise ValueError("Unidade obrigatória")
        self.unidade = unidade
        self.adj = {v: {} for v in vertices}
        self.indisponiveis = set()

    @property
    def vertices(self):
        return tuple(self.adj)

    def verificar_vertice(self, v):
        if v not in self.adj:
            raise ValueError(f"Vértice desconhecido: {v}")

    @staticmethod
    def verificar_peso(peso):
        if isinstance(peso, bool) or not isinstance(peso, Real):
            raise ValueError("Peso deve ser numérico")
        if not isfinite(peso) or peso < 0:
            raise ValueError("Dijkstra exige peso finito e não negativo")
        return float(peso)

    def adicionar_aresta(self, u, v, peso):
        self.verificar_vertice(u)
        self.verificar_vertice(v)
        peso = self.verificar_peso(peso)
        if v in self.adj[u]:
            raise ValueError("Aresta já cadastrada; use atualizar_peso")
        self.adj[u][v] = peso

    def verificar_aresta(self, u, v):
        self.verificar_vertice(u)
        self.verificar_vertice(v)
        if v not in self.adj[u]:
            raise ValueError(f"Aresta inexistente: {u} -> {v}")

    def atualizar_peso(self, u, v, peso):
        self.verificar_aresta(u, v)
        self.adj[u][v] = self.verificar_peso(peso)

    def bloquear_trecho(self, u, v):
        self.verificar_aresta(u, v)
        self.indisponiveis.add((u, v))

    def restaurar_trecho(self, u, v):
        self.verificar_aresta(u, v)
        self.indisponiveis.discard((u, v))

    def remover_aresta(self, u, v):
        self.verificar_aresta(u, v)
        del self.adj[u][v]
        self.indisponiveis.discard((u, v))


class DijkstraRouter:
    """Recalcula sobre o estado atual do grafo; (inf, []) significa sem rota."""
    def __init__(self, grafo):
        self.grafo = grafo
        self.ultima_trilha = []

    def calcular_menor_caminho(self, origem, destino, bloqueios=None,
                               arestas_bloqueadas=None):
        g = self.grafo
        g.verificar_vertice(origem)
        g.verificar_vertice(destino)
        bloqueios = set(bloqueios or ())
        arestas_bloqueadas = set(arestas_bloqueadas or ()) | g.indisponiveis
        for v in bloqueios:
            g.verificar_vertice(v)
        for u, v in arestas_bloqueadas:
            g.verificar_aresta(u, v)
        # Falha fechada mesmo se alguém editar adj diretamente.
        for u, vizinhos in g.adj.items():
            for v, peso in vizinhos.items():
                g.verificar_vertice(v)
                g.verificar_peso(peso)
        self.ultima_trilha = []
        if origem in bloqueios or destino in bloqueios:
            return inf, []
        dist = {v: inf for v in g.vertices}
        pred = {v: None for v in g.vertices}
        dist[origem] = 0.0
        heap = [(0.0, origem)]
        while heap:
            d, u = heappop(heap)
            if d != dist[u]:
                continue
            self.ultima_trilha.append((u, d))
            if u == destino:
                break
            for v, peso in g.adj[u].items():
                if v in bloqueios or (u, v) in arestas_bloqueadas:
                    continue
                candidato = d + peso
                if candidato < dist[v]:
                    dist[v], pred[v] = candidato, u
                    heappush(heap, (candidato, v))
        if dist[destino] == inf:
            return inf, []
        caminho, atual = [], destino
        while atual is not None:
            caminho.append(atual)
            atual = pred[atual]
        return dist[destino], list(reversed(caminho))


def validar_rota(g, resultado, origem, destino, bloqueios=(), arestas_bloqueadas=()):
    custo, rota = resultado
    if custo == inf:
        assert rota == []
        return
    assert rota[0] == origem and rota[-1] == destino
    assert len(rota) == len(set(rota))
    assert not set(rota).intersection(bloqueios)
    bloqueadas = set(arestas_bloqueadas) | g.indisponiveis
    soma = 0.0
    for u, v in zip(rota, rota[1:]):
        assert v in g.adj[u] and (u, v) not in bloqueadas
        soma += g.adj[u][v]
    assert isclose(soma, custo, rel_tol=1e-12, abs_tol=1e-12)


def deve_rejeitar(funcao, *args, **kwargs):
    try:
        funcao(*args, **kwargs)
    except ValueError:
        return
    raise AssertionError("Entrada inválida foi aceita")


def mostrar(nome, resultado, unidade):
    custo, rota = resultado
    if custo == inf:
        print(f"{nome}: SEM ROTA — nenhuma alternativa disponível")
    else:
        print(f"{nome}: {custo:.2f} {unidade} | " + " -> ".join(rota))


## Modelos de materiais e veículos

In [2]:
from collections import deque


class GrafoLogistico(GrafoDirigido):
    def __init__(self, vertices, camada, unidade="m"):
        if camada not in {"materiais", "veiculos"}:
            raise ValueError("Camada deve ser materiais ou veiculos")
        super().__init__(vertices, unidade)
        self.camada = camada

    def menor_caminho(self, origem, destino, **restricoes):
        return DijkstraRouter(self).calcular_menor_caminho(origem, destino, **restricoes)

    def ordem_topologica(self):
        # Topologia estrutural completa, inclusive arestas temporariamente bloqueadas.
        grau = {v: 0 for v in self.vertices}
        for vizinhos in self.adj.values():
            for v in vizinhos:
                grau[v] += 1
        fila = deque(v for v in self.vertices if grau[v] == 0)
        ordem = []
        while fila:
            u = fila.popleft()
            ordem.append(u)
            for v in self.adj[u]:
                grau[v] -= 1
                if grau[v] == 0:
                    fila.append(v)
        if len(ordem) != len(self.vertices):
            raise ValueError("Grafo contém ciclo dirigido; não admite ordem topológica")
        return ordem


def criar_fluxo_materiais():
    arestas = [
        ("RECEB_COPOS", "ESTOQUE_COPOS", 8),
        ("ESTOQUE_COPOS", "MAG-100_Copos", 6),
        ("MAG-100_Copos", "E1_DISPENSA", 1),
        ("RECEB_AGUA", "TK-101_Agua", 10),
        ("TK-101_Agua", "E2_ENVASE", 2.7),
        ("RECEB_TAMPAS", "ESTOQUE_TAMPAS", 7),
        ("ESTOQUE_TAMPAS", "E3_TAMPA", 5),
        ("E1_DISPENSA", "E2_ENVASE", 0.6),
        ("E2_ENVASE", "E3_TAMPA", 0.6),
        ("E3_TAMPA", "E4_SELAGEM", 0.6),
        ("E4_SELAGEM", "E5_EJECAO", 0.6),
        ("E5_EJECAO", "EST-500_Saida", 0.8),
        ("EST-500_Saida", "EMBALAGEM", 3),
        ("EMBALAGEM", "ESTOQUE_PRODUTO", 5),
        ("ESTOQUE_PRODUTO", "DOCA_EXPEDICAO", 8),
    ]
    nos = list(dict.fromkeys(v for u, v, w in arestas for v in (u, v)))
    g = GrafoLogistico(nos, "materiais")
    for aresta in arestas:
        g.adicionar_aresta(*aresta)
    return g


def criar_circulacao_veiculos(unidade="m"):
    # Modelo de carrinho/veículo interno. Não há vias através dos cilindros.
    arestas = [
        ("PORTARIA", "RECEBIMENTO", 20),
        ("RECEBIMENTO", "CORREDOR_A", 10),
        ("CORREDOR_A", "RECEBIMENTO", 10),
        ("CORREDOR_A", "ABASTECIMENTO", 8),
        ("ABASTECIMENTO", "CORREDOR_A", 8),
        ("CORREDOR_A", "CORREDOR_B", 12),
        ("CORREDOR_B", "CORREDOR_A", 12),
        ("CORREDOR_B", "ESTOQUE_FINAL", 8),
        ("ESTOQUE_FINAL", "CORREDOR_B", 8),
        ("ESTOQUE_FINAL", "DOCA", 10),
        ("DOCA", "SAIDA", 15),
        ("SAIDA", "PORTARIA", 10),
        ("RECEBIMENTO", "VIA_EXTERNA", 30),
        ("VIA_EXTERNA", "DOCA", 35),
    ]
    nos = list(dict.fromkeys(v for u, v, w in arestas for v in (u, v)))
    g = GrafoLogistico(nos, "veiculos", unidade)
    for u, v, distancia in arestas:
        if unidade == "m":
            custo = distancia
        elif unidade == "min":
            # Via externa: 10 m/min. Corredores: 2 m/min (hipóteses).
            externa = u == "VIA_EXTERNA" or v == "VIA_EXTERNA"
            custo = distancia / (10.0 if externa else 2.0)
        else:
            raise ValueError("Unidade de demonstração não suportada")
        g.adicionar_aresta(u, v, custo)
    return g


materiais = criar_fluxo_materiais()
veiculos = criar_circulacao_veiculos()
ordem = materiais.ordem_topologica()
print("Ordem topológica de materiais:", " -> ".join(ordem))
for inicio in ("RECEB_COPOS", "RECEB_AGUA", "RECEB_TAMPAS"):
    mostrar(inicio + " até expedição", materiais.menor_caminho(inicio, "DOCA_EXPEDICAO"), "m")
mostrar("Veículo: recebimento até doca", veiculos.menor_caminho("RECEBIMENTO", "DOCA"), "m")

# Validação estrutural da rota do copo: cinco operações obrigatórias em ordem.
rota_copo = materiais.menor_caminho("RECEB_COPOS", "DOCA_EXPEDICAO")
obrigatorias = ["E1_DISPENSA", "E2_ENVASE", "E3_TAMPA", "E4_SELAGEM", "E5_EJECAO"]
assert [v for v in rota_copo[1] if v in obrigatorias] == obrigatorias
assert isclose(rota_copo[0], 34.2)
validar_rota(materiais, rota_copo, "RECEB_COPOS", "DOCA_EXPEDICAO")

# A convergência de arestas não prova disponibilidade simultânea de insumos.
def liberar_lote(disponiveis):
    exigidos = {"copos", "agua", "tampas", "ar_comprimido"}
    return exigidos.issubset(set(disponiveis))

assert liberar_lote({"copos", "agua", "tampas", "ar_comprimido"})
assert not liberar_lote({"copos", "agua", "ar_comprimido"})
print("Disponibilidade conjunta de insumos: verificação booleana demonstrada.")


Ordem topológica de materiais: RECEB_COPOS -> RECEB_AGUA -> RECEB_TAMPAS -> ESTOQUE_COPOS -> TK-101_Agua -> ESTOQUE_TAMPAS -> MAG-100_Copos -> E1_DISPENSA -> E2_ENVASE -> E3_TAMPA -> E4_SELAGEM -> E5_EJECAO -> EST-500_Saida -> EMBALAGEM -> ESTOQUE_PRODUTO -> DOCA_EXPEDICAO
RECEB_COPOS até expedição: 34.20 m | RECEB_COPOS -> ESTOQUE_COPOS -> MAG-100_Copos -> E1_DISPENSA -> E2_ENVASE -> E3_TAMPA -> E4_SELAGEM -> E5_EJECAO -> EST-500_Saida -> EMBALAGEM -> ESTOQUE_PRODUTO -> DOCA_EXPEDICAO
RECEB_AGUA até expedição: 31.30 m | RECEB_AGUA -> TK-101_Agua -> E2_ENVASE -> E3_TAMPA -> E4_SELAGEM -> E5_EJECAO -> EST-500_Saida -> EMBALAGEM -> ESTOQUE_PRODUTO -> DOCA_EXPEDICAO
RECEB_TAMPAS até expedição: 30.00 m | RECEB_TAMPAS -> ESTOQUE_TAMPAS -> E3_TAMPA -> E4_SELAGEM -> E5_EJECAO -> EST-500_Saida -> EMBALAGEM -> ESTOQUE_PRODUTO -> DOCA_EXPEDICAO
Veículo: recebimento até doca: 40.00 m | RECEBIMENTO -> CORREDOR_A -> CORREDOR_B -> ESTOQUE_FINAL -> DOCA
Disponibilidade conjunta de insumos: verificaçã

## Investigações e testes automatizados

In [3]:
# Investigações adaptadas: estoques, área indisponível, sentidos, tempo e qualidade.
for inicio, estoque in [("RECEB_COPOS", "ESTOQUE_COPOS"),
                        ("RECEB_AGUA", "TK-101_Agua"),
                        ("RECEB_TAMPAS", "ESTOQUE_TAMPAS")]:
    resultado = materiais.menor_caminho(inicio, estoque)
    validar_rota(materiais, resultado, inicio, estoque)
    assert resultado[0] < inf
    mostrar("Recebimento até " + estoque, resultado, "m")

# Falha da armazenagem final: nenhuma rota deve pular o estoque.
m = criar_fluxo_materiais()
m.bloquear_trecho("EMBALAGEM", "ESTOQUE_PRODUTO")
assert m.menor_caminho("RECEB_COPOS", "DOCA_EXPEDICAO") == (inf, [])
m.restaurar_trecho("EMBALAGEM", "ESTOQUE_PRODUTO")
assert isclose(m.menor_caminho("RECEB_COPOS", "DOCA_EXPEDICAO")[0], 34.2)
for estacao in obrigatorias:
    assert m.menor_caminho("RECEB_COPOS", "DOCA_EXPEDICAO", bloqueios={estacao}) == (inf, [])
assert m.menor_caminho("DOCA_EXPEDICAO", "RECEB_COPOS") == (inf, [])
indice = {v: i for i, v in enumerate(m.ordem_topologica())}
assert all(indice[u] < indice[v] for u in m.vertices for v in m.adj[u])

# Sentido único: remover B -> A preserva A -> B; o anel ainda fecha via saída.
v = criar_circulacao_veiculos()
v.remover_aresta("CORREDOR_B", "CORREDOR_A")
assert "CORREDOR_A" not in v.adj["CORREDOR_B"]
assert "CORREDOR_B" in v.adj["CORREDOR_A"]
ciclo = ["PORTARIA", "RECEBIMENTO", "CORREDOR_A", "CORREDOR_B",
         "ESTOQUE_FINAL", "DOCA", "SAIDA", "PORTARIA"]
assert all(b in v.adj[a] for a, b in zip(ciclo, ciclo[1:]))
deve_rejeitar(v.ordem_topologica)  # Circulação tem ciclos legítimos.

# Caminho em metros difere do caminho em minutos: cada grafo mantém sua unidade.
por_distancia = veiculos.menor_caminho("RECEBIMENTO", "DOCA")
por_tempo = criar_circulacao_veiculos("min").menor_caminho("RECEBIMENTO", "DOCA")
assert por_distancia == (40.0, ["RECEBIMENTO", "CORREDOR_A", "CORREDOR_B", "ESTOQUE_FINAL", "DOCA"])
assert por_tempo == (6.5, ["RECEBIMENTO", "VIA_EXTERNA", "DOCA"])
mostrar("Ótimo por distância", por_distancia, "m")
mostrar("Ótimo por tempo", por_tempo, "min")
desvio = veiculos.menor_caminho("RECEBIMENTO", "DOCA", bloqueios={"CORREDOR_B"})
assert desvio == (65.0, ["RECEBIMENTO", "VIA_EXTERNA", "DOCA"])
validar_rota(veiculos, desvio, "RECEBIMENTO", "DOCA", {"CORREDOR_B"})
assert veiculos.menor_caminho("RECEBIMENTO", "DOCA", bloqueios={"CORREDOR_B", "VIA_EXTERNA"}) == (inf, [])

# Quarentena opcional, sem alterar o fluxo-base: única saída depende da qualidade.
q = GrafoLogistico(["RECEBIMENTO", "QUARENTENA", "ESTOQUE_LIBERADO"], "materiais")
q.adicionar_aresta("RECEBIMENTO", "QUARENTENA", 3)
q.adicionar_aresta("QUARENTENA", "ESTOQUE_LIBERADO", 2)
q.bloquear_trecho("QUARENTENA", "ESTOQUE_LIBERADO")
assert q.menor_caminho("RECEBIMENTO", "ESTOQUE_LIBERADO") == (inf, [])
def avaliar_lote(aprovado):
    if not isinstance(aprovado, bool):
        raise ValueError("Aprovação deve ser booleana")
    if aprovado:
        q.restaurar_trecho("QUARENTENA", "ESTOQUE_LIBERADO")
    else:
        q.bloquear_trecho("QUARENTENA", "ESTOQUE_LIBERADO")
avaliar_lote(True)
assert q.menor_caminho("RECEBIMENTO", "ESTOQUE_LIBERADO")[0] == 5.0
avaliar_lote(False)
assert q.menor_caminho("RECEBIMENTO", "ESTOQUE_LIBERADO") == (inf, [])
deve_rejeitar(avaliar_lote, "aprovado")
deve_rejeitar(m.menor_caminho, "PORTARIA", "DOCA_EXPEDICAO")  # Camadas distintas.
deve_rejeitar(GrafoLogistico, ["A"], "camada_invalida")
deve_rejeitar(m.atualizar_peso, "RECEB_COPOS", "ESTOQUE_COPOS", -1)
print("TESTES 18 APROVADOS: sequência, custos, DAG, bloqueios, sentidos, unidades e quarentena.")


Recebimento até ESTOQUE_COPOS: 8.00 m | RECEB_COPOS -> ESTOQUE_COPOS
Recebimento até TK-101_Agua: 10.00 m | RECEB_AGUA -> TK-101_Agua
Recebimento até ESTOQUE_TAMPAS: 7.00 m | RECEB_TAMPAS -> ESTOQUE_TAMPAS
Ótimo por distância: 40.00 m | RECEBIMENTO -> CORREDOR_A -> CORREDOR_B -> ESTOQUE_FINAL -> DOCA
Ótimo por tempo: 6.50 min | RECEBIMENTO -> VIA_EXTERNA -> DOCA
TESTES 18 APROVADOS: sequência, custos, DAG, bloqueios, sentidos, unidades e quarentena.
